In [1]:

import sys
from pathlib import Path

import nibabel as nib
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# Project paths
ROOT = Path(r"D:\PS07_CT2MRI")
VOLUMES = ROOT / "dataset" / "volumes"

# Import official normalization functions
sys.path.insert(0, str(ROOT / "dataset"))
from normalise import normalise_ct, normalise_mri

# Select the first official training patient
import pandas as pd

train_df = pd.read_csv(ROOT / "train_patients.csv")
patient_id = train_df.iloc[0]["id"]

print("Testing patient:", patient_id)


class CTMRISliceDataset(Dataset):

    def __init__(self, patient_id):

        folder = VOLUMES / patient_id

        # Load one patient's volumes
        ct = nib.load(str(folder / "ct.nii.gz")).get_fdata(
            dtype=np.float32
        )
        mr = nib.load(str(folder / "mr.nii.gz")).get_fdata(
            dtype=np.float32
        )
        mask = nib.load(str(folder / "mask.nii.gz")).get_fdata(
            dtype=np.float32
        )

        # Apply official normalization
        self.ct = normalise_ct(ct)
        self.mr = normalise_mri(mr, mask)
        self.mask = mask

        # Use slices with foreground mask
        self.valid_slices = [
            z for z in range(1, ct.shape[2] - 1)
            if np.any(mask[:, :, z] > 0)
        ]

        print("Valid slices:", len(self.valid_slices))

    def __len__(self):
        return len(self.valid_slices)

    def __getitem__(self, index):

        z = self.valid_slices[index]

        # Three neighbouring CT slices = 3 input channels
        ct_input = np.stack([
            self.ct[:, :, z - 1],
            self.ct[:, :, z],
            self.ct[:, :, z + 1]
        ], axis=0)

        # Central MRI slice = target
        mr_target = self.mr[:, :, z][None, :, :]

        # Corresponding mask
        mask_slice = self.mask[:, :, z][None, :, :]

        return (
            torch.from_numpy(ct_input.copy()),
            torch.from_numpy(mr_target.copy()),
            torch.from_numpy(mask_slice.copy())
        )


# Test using one training patient only
dataset = CTMRISliceDataset(patient_id)

loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
    num_workers=0
)

# Inspect one batch
ct_batch, mr_batch, mask_batch = next(iter(loader))

print("\nDataLoader test successful!")

print("CT input shape:", ct_batch.shape)
print("MRI target shape:", mr_batch.shape)
print("Mask shape:", mask_batch.shape)

print("CT range:", ct_batch.min().item(),
      "to", ct_batch.max().item())

print("MRI range:", mr_batch.min().item(),
      "to", mr_batch.max().item())

print("Device available:", torch.cuda.is_available())


Testing patient: P1046
Valid slices: 179

DataLoader test successful!
CT input shape: torch.Size([2, 3, 192, 224])
MRI target shape: torch.Size([2, 1, 192, 224])
Mask shape: torch.Size([2, 1, 192, 224])
CT range: 0.0 to 0.9113333225250244
MRI range: 0.0 to 1.0
Device available: True
